<a href="https://colab.research.google.com/github/jackychh7878/Colab_AI_Project/blob/main/Q%26A_with_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answer Application
The goal of Question Answering is to find the answer to a question given a question and an accompanying context. The predicted answer will be either a span of text from the context or an empty string (indicating the question cannot be answered from the context).

Dataset: https://www.kaggle.com/datasets/stanfordu/stanford-question-answering-dataset?select=dev-v1.1.json

The Stanford Question Answering Dataset (SQuAD) is a reading comprehension dataset consisting of questions posed by crowdworkers on a set of Wikipedia articles. The answer to every question is a segment of text, or span, from the corresponding reading passage. There are 100,000+ question-answer pairs on 500+ articles.

In [ ]:
!pip install simpletransformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 8.4 MB/s eta 0:00:00

# Step 1: Import training data

In [ ]:
import json
with open("/content/train.json", "r") as read_file:
  train = json.load(read_file)

with open("/content/dev.json", "r") as read_file:
  test = json.load(read_file)

In [ ]:
train['data'][0]

{'title': 'University_of_Notre_Dame',
 'paragraphs': [{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
   'qas': [{'answers': [{'answer_start': 515,
       'text': 'Saint Bernadette Soubirous'}],
     'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
     'id': '5733be284776f41900661182'},
    {'ans

In [ ]:
test

{'data': [{'title': 'Super_Bowl_50',
   'paragraphs': [{'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
     'qas': [{'answers': [{'answer_start': 177, 'text': 'Denver Broncos'},
        {'answer_start': 177, 'text': 'Denver Broncos'},
        {'answer_star

# Step 2: Config the model

In [ ]:
from simpletransformers.question_answering import QuestionAnsweringModel, QuestionAnsweringArgs

model_type="bert"
model_name= "bert-base-cased"

# configure model

model_args = QuestionAnsweringArgs()
model_args.train_batch_size = 16
model_args.evaluate_during_training = True
model_args.n_best_size = 3
model_args.num_train_epochs = 5


# Training config
train_args = {
    "reprocess_input_data": True,
    "overwrite_output_dir": True,
    "use_cached_eval_features": True,
    "output_dir": f"outputs/{model_type}",
    "best_model_dir": f"outputs/{model_type}/best_model",
    "evaluate_during_training": True,
    "max_seq_length": 128,
    "num_train_epochs": 5,
    "evaluate_during_training_steps": 1000,
    "wandb_project": "Question Answer Application",
    "wandb_kwargs": {"name": model_name},
    "save_model_every_epoch": False,
    "save_eval_checkpoints": False,
    "n_best_size":3,
    "train_batch_size": 128,
    "eval_batch_size": 64,

}

In [ ]:
model = QuestionAnsweringModel(model_type, model_name, args=train_args)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

# Step 3: Train the model using wandb

In [ ]:
model.train_model(train['data'][0]['paragraphs'], eval_data=test['data'][0]['paragraphs'])

add example index and unique id: 100%|██████████| 269/269 [00:00<00:00, 341588.79it/s]


Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

correct,█▁▁▁▁
eval_loss,█▅▃▁▁
global_step,▁▃▅▆█
incorrect,█▂▁▁▁
similar,▁▇███
train_loss,█▅▄▃▁
correct,0
eval_loss,-0.39353
global_step,15
incorrect,0
similar,810


Running Epoch 1 of 5:   0%|          | 0/3 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/13 [00:00<?, ?it/s]

...

Running Epoch 4 of 5:   0%|          | 0/3 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/13 [00:00<?, ?it/s]

Running Epoch 5 of 5:   0%|          | 0/3 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/13 [00:00<?, ?it/s]

(15,
 {'global_step': [3, 6, 9, 12, 15],
  'correct': [0, 4, 5, 10, 19],
  'similar': [805, 786, 769, 737, 700],
  'incorrect': [5, 20, 36, 63, 91],
  'train_loss': [2.990084171295166,
   3.066856861114502,
   2.26904296875,
   1.983135461807251,
   2.131235122680664],
  'eval_loss': [-0.42882361778846156,
   -0.5432316706730769,
   -0.668212890625,
   -0.7881798377403846,
   -0.8384164663461539]})

# Step 4: Evaluate the model

In [ ]:
# Evaluate the model
result, texts = model.eval_model(test['data'][0]['paragraphs'])

Running Evaluation:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
result

{'correct': 19,
 'similar': 700,
 'incorrect': 91,
 'eval_loss': -0.8384164663461539}

# Step 5: Interencing

In [ ]:
# Make predictions with the model
to_predict = [
    {
        "context": "The first phase of Eddy Street Commons, a $215 million development located adjacent to the University of Notre Dame campus and funded by the university, broke ground on June 3, 2008. The Eddy Street Commons drew union protests when workers hired by the City of South Bend to construct the public parking garage picketed the private work site after a contractor hired non-union workers. The developer, Kite Realty out of Indianapolis, has made agreements with major national chains rather than local businesses, a move that has led to criticism from alumni and students.",
        "qas": [
            {
                "question": "How much is Eddy Street Commons at Notre Dame expected to cost?",
                "id": "0",
            }
        ],
    }
]

answers, probabilities = model.predict(to_predict)

print(answers)

add example index and unique id: 100%|██████████| 1/1 [00:00<00:00, 4355.46it/s]


Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '0', 'answer': ['$215 million', '$215 million development located adjacent to the University of Notre Dame campus and funded by the university, broke ground on June 3, 2008', 'June 3, 2008']}]


In [ ]:
# Make predictions with the model
to_predict = [
    {
        "context": "Notre Dame teams are known as the Fighting Irish. They compete as a member of the National Collegiate Athletic Association (NCAA) Division I, primarily competing in the Atlantic Coast Conference (ACC) for all sports since the 2013–14 school year. The Fighting Irish previously competed in the Horizon League from 1982-83 to 1985-86, and again from 1987-88 to 1994-95, and then in the Big East Conference through 2012–13. Men's sports include baseball, basketball, crew, cross country, fencing, football, golf, ice hockey, lacrosse, soccer, swimming & diving, tennis and track & field; while women's sports include basketball, cross country, fencing, golf, lacrosse, rowing, soccer, softball, swimming & diving, tennis, track & field and volleyball. The football team competes as an Football Bowl Subdivision (FBS) Independent since its inception in 1887. Both fencing teams compete in the Midwest Fencing Conference, and the men's ice hockey team competes in Hockey East.",
        "qas": [
            {
                "question": "Which league did Notre Dame Fighting Irish teams participate in in 1982?",
                "id": "0",
            }
        ],
    }
]
answers, probabilities = model.predict(to_predict)

print(answers)

add example index and unique id: 100%|██████████| 1/1 [00:00<00:00, 4911.36it/s]


Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '0', 'answer': ['', 'Horizon League']}]
